In [2]:
import torch
import torch.nn as nn

## 1. Tensors & Broadcasting

In [4]:
x = torch.randn(2,8,64) # [B,T,C]
b = torch.randn(8,64) # [8,64]
s = torch.randn(64)  # [ 64]

print((x+b).shape)
print((x+s).shape)
print((b+s).shape)

torch.Size([2, 8, 64])
torch.Size([2, 8, 64])
torch.Size([8, 64])


In PyTorch, broadcasting allows tensors with different shapes to be added together by automatically expanding dimensions of size **1** or missing dimensions.

No real data is copied—PyTorch only *pretends* the smaller tensor is expanded.

---

## Given shapes

- x = [B, T, C]
- b = [T, C]
- s = [C]

---

## Rules (simple)

When adding tensors:
- If a dimension matches → keep it
- If a dimension is missing or is 1 → expand it to match
- Otherwise → error

---

## Results

### 1. x + b → [B, T, C]
- b is treated as [1, T, C]
- Broadcast across batch dimension
- Same position-wise pattern applied to all batches

---

### 2. x + s → [B, T, C]
- s is treated as [1, 1, C]
- Broadcast across batch and time dimensions
- Same feature-wise shift applied to all tokens

---

### 3. b + s → [T, C]
- s is treated as [1, C]
- Broadcast across time dimension
- Each position keeps its own vector, with shared feature adjustment

---

## Transformer intuition

In a Transformer:
- [T, C] → position-wise information
- [C] → feature-wise global parameters
- [B, T, C] → full sequence representation

Broadcasting is what allows compact parameters to operate over full sequences without loops.

---

## Key idea

Broadcasting = “expand dimensions of size 1 to match shapes for element-wise operations”

In [6]:
# critical for attention mask
mask = torch.ones(1,1,8,8) # [1,1,T,T]
attn = torch.ones(2,4,8,8) # [B,H,T,T]

print(mask+attn)

tensor([[[[2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.]],

         [[2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.]],

         [[2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2., 2., 2., 2.],
          [2., 2., 2., 2., 2

In [7]:
# transpose — swap 2 dims only
x = torch.randn(2, 8, 64) # [B, T, C]
x.transpose(1, 2)   # → [2, 64, 8]  swap T and C | swap dimension 1 and dimension 2

# permute — full reorder  | re-arrange
x = torch.randn(2, 4, 8, 16)  # [B, H, T, head_dim]
x.permute(0, 2, 1, 3)          # [B, T, H, head_dim]

# contiguous() — needed after permute before view/reshape
x.permute(0,2,1,3).contiguous().view(2, 8, -1)
# → [B, T, H*head_dim]  = [2, 8, 64]

tensor([[[ 1.2036,  0.3332,  0.1604,  ...,  1.1641,  0.2628,  1.2808],
         [-1.1914, -0.2543, -1.2658,  ..., -0.0673,  1.2298,  0.3207],
         [ 0.2002, -0.3451, -0.0971,  ..., -0.1920,  1.3401, -1.1224],
         ...,
         [-0.9451,  0.5030,  1.1062,  ...,  0.2286,  0.2302, -2.4487],
         [-0.0213,  0.1404, -1.4423,  ...,  1.8055,  1.2139, -0.0440],
         [-0.7459, -2.2482, -0.0239,  ...,  0.4176,  0.9748, -2.4380]],

        [[ 0.4927, -0.9630,  0.2383,  ..., -0.3114, -0.4782, -0.1432],
         [-1.0965, -0.4392,  0.2044,  ...,  0.4890, -1.4788, -0.1662],
         [ 0.1922,  0.0652,  0.2690,  ..., -0.0087, -0.3506, -0.0989],
         ...,
         [ 0.2055, -0.6428, -0.8693,  ...,  0.4863,  0.7791,  0.4070],
         [-0.8168, -0.3841,  0.2750,  ...,  0.2074,  0.3086,  0.8541],
         [-0.7067, -1.3661, -1.1879,  ...,  0.6539, -0.1982, -0.4980]]])

# transpose, permute, contiguous

These are **shape-reordering tools** used heavily in Transformers, especially in multi-head attention.

---

## 1. transpose — swap 2 dimensions

Used when you only need to swap **two axes**.

Example:
- x = [B, T, C]
- transpose(1, 2) → [B, C, T]

### Meaning (Transformer POV)
You are just reinterpreting:
> “time dimension vs feature dimension”

Common use:
- preparing matrices for dot-product attention

---

## 2. permute — reorder all dimensions

Used when you need a full re-layout of tensor axes.

Example:
- [B, H, T, head_dim]
→ [B, T, H, head_dim]

### Meaning (Transformer POV)

You are changing how attention is viewed:

> from “heads-first view” → to “token-first view”

This is important when merging heads back into tokens.

---

## 3. contiguous() — memory fix after permute

After `permute`, tensor is usually:

> not stored in contiguous memory layout

So operations like `.view()` may fail or behave incorrectly.

### contiguous() does:
- reorganizes memory layout physically
- makes tensor safe for `.view()`

---

## 4. view() — reshape without changing data

After making tensor contiguous:

Example:
- [B, T, H, head_dim]
→ [B, T, H * head_dim]

This is where **multi-head attention is merged back**.

---

## Transformer connection (very important)

In a :contentReference[oaicite:0]{index=0}:

### Multi-head attention flow:

1. Start:
   - [B, H, T, head_dim]

2. permute:
   - [B, T, H, head_dim]

3. merge heads:
   - [B, T, C] where C = H × head_dim

---

## Key intuition

- transpose → swap 2 axes
- permute → reorder full structure
- contiguous → fix memory layout
- view → reshape into final representation

---

## One-line summary

These operations are not about math—they are about:
> changing how the same tensor is *viewed* so it fits attention computations efficiently

In [10]:
# view
x = torch.arange(12)
print(x)
print(x.shape)

y = x.view(3,4)  # same-memory , new shape
print(y)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
torch.Size([12])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


## 2. Linear Layer

### 🖊 Pen-paper: what it actually does

y = x ·  Wᵀ + b

- **x**: `[B, T, in_features]` → e.g. `[2, 8, 64]`  

- **W**: `[out_features, in_features]` → e.g. `[128, 64]`  

- **b**: `[out_features]` → e.g. `[128]`  

- **y**: `[B, T, out_features]` → e.g. `[2, 8, 128]`

In [5]:
# pyTorch nn.Linear
layer = nn.Linear(64,128) # in=64 , out=128
# layer.weight.shape → [128, 64]  (out, in)
# layer.bias.shape   → [128]

x = torch.randn(2,8,64) # [B, T, C]
y = layer(x)  # [B, T, 128]

print(x.shape)
print(y.shape)

torch.Size([2, 8, 64])
torch.Size([2, 8, 128])


In [16]:
# eg :-

layer = nn.Linear(3, 5)
x = torch.randn(1, 3) 
y = layer(x)

print(x)
print(y)
print("---"*20)
print(layer.weight)
print(layer.bias)

print("---"*20)
# In-place check of parameters
for name, p in layer.named_parameters():
    print(name, p.shape)

tensor([[-0.7512, -1.3646, -1.7298]])
tensor([[ 1.3396,  0.7467,  0.2835,  0.3717, -1.0671]],
       grad_fn=<AddmmBackward0>)
------------------------------------------------------------
Parameter containing:
tensor([[-0.4948, -0.2301, -0.3009],
        [-0.2604, -0.3988,  0.2563],
        [-0.1409,  0.1157, -0.0383],
        [-0.1424,  0.2699, -0.2621],
        [-0.4981,  0.5329,  0.4049]], requires_grad=True)
Parameter containing:
tensor([ 0.1334,  0.4503,  0.2693,  0.1797, -0.0137], requires_grad=True)
------------------------------------------------------------
weight torch.Size([5, 3])
bias torch.Size([5])


In [15]:
layer = nn.Linear(3, 5,bias=False)  # do not add bias
x = torch.randn(1, 3) 
y = layer(x)

print(x)
print(y)
print("---"*20)
print(layer.weight)
print(layer.bias)  

tensor([[0.7122, 0.9046, 1.0124]])
tensor([[ 0.9618, -0.4833,  0.0187,  0.7936,  0.1407]], grad_fn=<MmBackward0>)
------------------------------------------------------------
Parameter containing:
tensor([[ 0.4807,  0.2114,  0.4230],
        [-0.0338, -0.2478, -0.2322],
        [-0.1509,  0.2414, -0.0911],
        [-0.0930,  0.4949,  0.4070],
        [-0.0839,  0.0469,  0.1560]], requires_grad=True)
None


In [34]:
# Autograd

x = torch.tensor(2.0,requires_grad=True)
print(x)

tensor(2., requires_grad=True)


In [35]:
y = x * x
print(y)

tensor(4., grad_fn=<MulBackward0>)


In [36]:
y.backward()
print(x.grad)

tensor(4.)


In [38]:
y2 = x*x + 5*x
y2.backward()
print(x.grad)  # x.grad accumulates gradients from every backward() call

tensor(22.)
